# 01 · California Housing на Spark

Регрессия на 20k строках California Housing. Сравниваем Linear, Ridge, Lasso, Decision Tree, Random Forest и Gradient-Boosted Trees. Данные загружаются штатным loader scikit-learn (он уже есть в проекте); очистка, split, обучение, оценка и CV выполняются на Spark ML.

Ресурсные настройки зафиксированы в ячейке и соответствуют текущему compose: драйвер 2 GB; один executor до 4 cores / 6 GB; `shuffle.partitions=8`; отчётные массивы ограничены 5 000 точками.

## 1. Среда, пути и SparkSession

Подключаем Spark Standalone, если порт master доступен. Вне docker-сети автоматически используется `local[2]`; версии дополнительных библиотек не устанавливаются.

In [ ]:
# Стек проекта: только уже установленные пакеты + PySpark из requirements.txt.
import os
import re
import json
import socket
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

SEED = 42
TASK = 'housing'
TITLE = 'California Housing · Spark ML'
HERO_METRIC = 'mae'
TARGET_UNIT = '$100k'
MAX_REPORT_ROWS = 5000       # лимитируем JSON, а не распределённое обучение
EDA_SAMPLE_ROWS = 12000
CV_FOLDS = 3
CV_SAMPLE_ROWS = 50000      # верхняя граница CV-набора для крупных датасетов
CV_MODELS = ['RandomForest', 'GradBoost']


def find_project_root(start=None):
    """Находим корень проекта по requirements.txt; в Docker обычно это /app."""
    p = Path(start or Path.cwd()).resolve()
    for parent in (p, *p.parents):
        if (parent / 'requirements.txt').exists():
            return parent
    return Path.cwd().resolve()


PROJECT = find_project_root()
MODEL_DIR = PROJECT / 'models' / 'regression' / TASK
ART_DIR = PROJECT / 'artifacts' / 'regression' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)


def select_spark_master():
    """Используем standalone-кластер, если он доступен; иначе — локальный Spark."""
    requested = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077').strip()
    if requested.startswith('spark://'):
        endpoint = requested[len('spark://'):]
        host, _, port_text = endpoint.rpartition(':')
        try:
            with socket.create_connection((host, int(port_text)), timeout=1.5):
                print(f'Spark master доступен: {requested}')
                return requested
        except Exception as exc:
            print(f'Spark master не доступен ({type(exc).__name__}: {exc}); используем local[2].')
            return 'local[2]'
    return requested or 'local[2]'


SPARK_MASTER = select_spark_master()
remote = SPARK_MASTER.startswith('spark://')
builder = (
    SparkSession.builder
    .appName(f'ml-regression-{TASK}')
    .master(SPARK_MASTER)
    .config('spark.driver.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.executor.memory', '6g' if remote else '2g')
    .config('spark.executor.memoryOverhead', '768m' if remote else '512m')
    .config('spark.executor.cores', '4' if remote else '2')
    .config('spark.cores.max', '4' if remote else '2')
    .config('spark.default.parallelism', '8' if remote else '4')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.ansi.enabled', 'false')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.fallback.enabled', 'true')
)
if Path('/opt/spark/events').is_dir():
    builder = (builder
        .config('spark.eventLog.enabled', 'true')
        .config('spark.eventLog.dir', 'file:///opt/spark/events'))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print('Project      :', PROJECT)
print('Spark        :', spark.version)
print('Master       :', spark.sparkContext.master)
print('Driver limit : 2 GB; executor limit :', '6 GB / 4 cores' if remote else '2 GB / 2 cores (local)')
print('Model dir    :', MODEL_DIR)
print('Artifacts dir:', ART_DIR)


## 2. Загрузка и очистка данных

Источник выбирается как в исходном ноутбуке. После загрузки данные приводятся к числовому типу и остаются в Spark DataFrame.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
housing_pdf = housing.frame.reset_index(drop=True)
TARGET = 'MedHouseVal'
FEATURES = [c for c in housing_pdf.columns if c != TARGET]
DATASET_NAME = 'sklearn.fetch_california_housing'

# Источник небольшой; после загрузки подготовка, split, обучение и метрики идут в Spark.
data_sdf = spark.createDataFrame(housing_pdf)
data_sdf = data_sdf.withColumn('_row_id', F.monotonically_increasing_id())
for c in FEATURES + [TARGET]:
    data_sdf = data_sdf.withColumn(c, F.col(c).cast('double'))
data_sdf = data_sdf.filter(
    F.col(TARGET).isNotNull() &
    F.col("MedInc").isNotNull() & F.col("HouseAge").isNotNull() & F.col("AveRooms").isNotNull() & F.col("AveBedrms").isNotNull() & F.col("Population").isNotNull() & F.col("AveOccup").isNotNull() & F.col("Latitude").isNotNull() & F.col("Longitude").isNotNull()
).select('_row_id', *FEATURES, TARGET)

data_sdf = data_sdf.cache()
N_SAMPLES = data_sdf.count()
print('Dataset:', DATASET_NAME)
print('Rows   :', N_SAMPLES)
print('Target :', TARGET, '(единица — $100,000)')
print('Features:', FEATURES)
print('Target statistics:')
data_sdf.select(TARGET).describe().show()


## 3. EDA и train/test split

В pandas преобразуется только ограниченная выборка для графиков; полное обучение остаётся в Spark. Разбиение фиксировано по `seed=42`.

In [ ]:
# Быстрая EDA: в pandas попадает только ограниченная выборка для графиков.
frac = min(1.0, max(0.15, (EDA_SAMPLE_ROWS * 1.5) / max(N_SAMPLES, 1)))
plot_pdf = (data_sdf.select(*FEATURES, TARGET)
            .sample(withReplacement=False, fraction=frac, seed=SEED)
            .limit(EDA_SAMPLE_ROWS)
            .toPandas())
if plot_pdf.empty:
    raise RuntimeError('После очистки датасет пустой — проверьте исходный CSV/схему.')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(plot_pdf[TARGET].dropna(), bins=50, edgecolor='white')
axes[0].set_title(f'{TARGET}: распределение')
axes[0].set_xlabel(TARGET)
axes[1].boxplot(plot_pdf[TARGET].dropna(), vert=False)
axes[1].set_title('Target: box-plot')
axes[1].set_xlabel(TARGET)

corr_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(plot_pdf[c])]
if corr_cols:
    corr = plot_pdf[corr_cols + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values()
    axes[2].barh(corr.index, corr.values)
    axes[2].axvline(0, color='black', linewidth=0.7)
    axes[2].set_title('Корреляции с target (выборка)')
else:
    axes[2].text(0.1, 0.5, 'Нет числовых признаков', transform=axes[2].transAxes)
    axes[2].set_axis_off()
plt.tight_layout()
plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight')
plt.show()

print('EDA sample:', len(plot_pdf), 'of', N_SAMPLES)
print('Target median (approx):', data_sdf.approxQuantile(TARGET, [0.5], 0.001)[0])

# Все дальнейшие операции обучения и оценки выполняются на Spark DataFrame.
train_df, test_df = data_sdf.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.cache()
test_df = test_df.cache()
N_TRAIN = train_df.count()
N_TEST = test_df.count()
if N_TRAIN < 100 or N_TEST < 20:
    raise ValueError(f'Недостаточно строк для обучения: train={N_TRAIN}, test={N_TEST}')
print(f'Train: {N_TRAIN:,} | test: {N_TEST:,} | split: 80/20')


## 4. Метрики и модели

Обучаем шесть вариантов. Деревья ограничены по числу деревьев/итераций и глубине. Метрики рассчитываются распределённо; на driver возвращаются только агрегированные числа.

In [ ]:
def make_estimator(name):
    """Умеренные размеры моделей, рассчитанные на worker 4 cores / 6 GB heap."""
    if name == 'Linear':
        return LinearRegression(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxIter=35, regParam=0.0, elasticNetParam=0.0, standardization=True,
        )
    if name == 'Ridge':
        return LinearRegression(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxIter=35, regParam=0.1, elasticNetParam=0.0, standardization=True,
        )
    if name == 'Lasso':
        return LinearRegression(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxIter=35, regParam=0.001, elasticNetParam=1.0, standardization=True,
        )
    if name == 'Huber':
        # Spark MLlib поддерживает Huber loss; отдельная библиотека не нужна.
        return LinearRegression(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxIter=35, regParam=0.01, elasticNetParam=0.0,
            loss='huber', epsilon=1.35, standardization=True,
        )
    if name == 'DecisionTree':
        return DecisionTreeRegressor(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxDepth=8, maxBins=48, minInstancesPerNode=5, seed=SEED,
        )
    if name == 'RandomForest':
        return RandomForestRegressor(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            numTrees=40, maxDepth=8, maxBins=48, minInstancesPerNode=5,
            subsamplingRate=0.8, seed=SEED,
        )
    if name == 'GradBoost':
        return GBTRegressor(
            featuresCol='features', labelCol=TARGET, predictionCol='prediction',
            maxIter=40, maxDepth=4, maxBins=48, stepSize=0.05,
            subsamplingRate=0.8, seed=SEED,
        )
    raise KeyError(f'Неизвестная модель: {name}')


def make_pipeline(name):
    stages = []
    # Store — категория для линейных моделей; деревья используют исходный числовой код.
    use_store_ohe = TASK == 'walmart' and 'Store' in FEATURES and name in {'Linear', 'Ridge'}
    if use_store_ohe:
        stages.append(StringIndexer(inputCol='Store', outputCol='Store_index', handleInvalid='keep'))
        stages.append(OneHotEncoder(
            inputCols=['Store_index'], outputCols=['Store_ohe'],
            handleInvalid='keep', dropLast=True,
        ))
        assembler_inputs = [c for c in FEATURES if c != 'Store'] + ['Store_ohe']
    else:
        assembler_inputs = list(FEATURES)
    stages.append(VectorAssembler(
        inputCols=assembler_inputs, outputCol='features', handleInvalid='error'))
    stages.append(make_estimator(name))
    return Pipeline(stages=stages)


def metrics_from_spark(prediction_df):
    """Метрики считаются распределённо; на driver возвращается только одна строка."""
    base = (prediction_df
        .select(F.col(TARGET).cast('double').alias('_y'),
                F.col('prediction').cast('double').alias('_p'))
        .withColumn('_err', F.col('_p') - F.col('_y'))
        .withColumn('_abs_err', F.abs(F.col('_p') - F.col('_y'))))
    # MAPE определяется только для ненулевых target; в этих датасетах target > 0.
    rel_expr = F.when(F.abs(F.col('_y')) > F.lit(1e-9), F.col('_abs_err') / F.abs(F.col('_y')))
    agg_exprs = [
        F.count('*').alias('n'),
        F.avg('_abs_err').alias('mae'),
        F.sqrt(F.avg(F.col('_err') * F.col('_err'))).alias('rmse'),
        (F.avg(rel_expr) * F.lit(100.0)).alias('mape'),
        F.avg('_err').alias('bias'),
        F.max('_abs_err').alias('max_error'),
        F.sum(F.col('_err') * F.col('_err')).alias('sse'),
        F.sum('_y').alias('sum_y'),
        F.sum(F.col('_y') * F.col('_y')).alias('sum_y2'),
        F.var_pop('_y').alias('var_y'),
        F.var_pop('_err').alias('var_err'),
    ]
    if TASK == 'walmart':
        log_err = (F.log(F.greatest(F.col('_p'), F.lit(0.0)) + F.lit(1.0)) -
                   F.log(F.greatest(F.col('_y'), F.lit(0.0)) + F.lit(1.0)))
        agg_exprs.append(F.sqrt(F.avg(log_err * log_err)).alias('rmsle'))
    row = base.agg(*agg_exprs).first().asDict()
    n = max(int(row['n'] or 0), 1)
    sst = float(row['sum_y2'] or 0.0) - float(row['sum_y'] or 0.0) ** 2 / n
    var_y = float(row['var_y'] or 0.0)
    mae = float(row['mae'] or 0.0)
    med = base.approxQuantile('_abs_err', [0.5], 0.001)
    result = {
        'mae': mae,
        'rmse': float(row['rmse'] or 0.0),
        'mape': float(row['mape'] or 0.0),
        'r2': float(1.0 - float(row['sse'] or 0.0) / sst) if sst > 1e-12 else 0.0,
        'medae': float(med[0]) if med else 0.0,
        'bias': float(row['bias'] or 0.0),
        'max_error': float(row['max_error'] or 0.0),
        'evs': float(1.0 - float(row['var_err'] or 0.0) / var_y) if var_y > 1e-12 else 0.0,
    }
    if TASK == 'walmart':
        result['rmsle'] = float(row.get('rmsle') or 0.0)
    return result


def predict_with(name, frame):
    if name == 'BaselineMedian':
        return frame.withColumn('prediction', F.lit(TRAIN_MEDIAN).cast('double'))
    return fitted[name].transform(frame)


# Колонки с базовой схемой: 6 сравниваемых моделей на Housing и Diamonds,
# 5 ML-моделей + медианный baseline для Walmart.
if TASK == 'housing':
    MODEL_NAMES = ['Linear', 'Ridge', 'Lasso', 'DecisionTree', 'RandomForest', 'GradBoost']
elif TASK == 'walmart':
    MODEL_NAMES = ['BaselineMedian', 'Linear', 'Ridge', 'DecisionTree', 'RandomForest', 'GradBoost']
else:
    MODEL_NAMES = ['Linear', 'Ridge', 'Huber', 'DecisionTree', 'RandomForest', 'GradBoost']

TRAIN_MEDIAN = float(train_df.approxQuantile(TARGET, [0.5], 0.001)[0]) if TASK == 'walmart' else None
results = {}
fitted = {}

for model_name in MODEL_NAMES:
    print(f'>>> {model_name}', flush=True)
    if model_name == 'BaselineMedian':
        pred_df = test_df.withColumn('prediction', F.lit(TRAIN_MEDIAN).cast('double'))
    else:
        fitted[model_name] = make_pipeline(model_name).fit(train_df)
        pred_df = fitted[model_name].transform(test_df)
    results[model_name] = metrics_from_spark(pred_df)
    print('   ', {k: round(v, 4) for k, v in results[model_name].items()})

metrics_pdf = pd.DataFrame(results).T
metric_order = ['mae', 'rmse'] + (['rmsle'] if TASK == 'walmart' else []) + ['mape', 'r2', 'medae', 'bias', 'max_error', 'evs']
metrics_pdf = metrics_pdf[[c for c in metric_order if c in metrics_pdf.columns]]
metrics_pdf = metrics_pdf.round(4)
metrics_pdf


## 5. Сравнение моделей, остатки и важность признаков

Основная метрика для выбора модели зависит от сценария. Отдельно сохраняются график остатков, actual-vs-predicted и сравнение ошибок.

In [ ]:
# Лучшие по основной метрике и RMSE могут различаться — сохраняем оба ранжирования.
best_name = min(results, key=lambda n: results[n][HERO_METRIC])
best_by_rmse = min(results, key=lambda n: results[n]['rmse'])
print(f'Лучшая по {HERO_METRIC}: {best_name} = {results[best_name][HERO_METRIC]:.4f}')
print(f'Лучшая по RMSE       : {best_by_rmse} = {results[best_by_rmse]["rmse"]:.4f}')
print('RMSE / MAE для победителя:', round(results[best_name]['rmse'] / max(results[best_name]['mae'], 1e-9), 3))
print('MAE / MedAE для победителя:', round(results[best_name]['mae'] / max(results[best_name]['medae'], 1e-9), 3))

best_pred_sdf = predict_with(best_name, test_df)
# Для диагностики на driver выгружаем только предсказание победителя; полный test не нужен для графиков.
plot_fraction = min(1.0, max(0.1, 7000.0 / max(N_TEST, 1) * 1.5))
best_plot_pdf = (best_pred_sdf.select('_row_id', *([c for c in ['Date', 'Store'] if c in test_df.columns]), TARGET, 'prediction')
                 .sample(withReplacement=False, fraction=plot_fraction, seed=SEED)
                 .limit(7000).toPandas())
best_plot_pdf['residual'] = best_plot_pdf['prediction'] - best_plot_pdf[TARGET]
best_plot_pdf['abs_error'] = best_plot_pdf['residual'].abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(best_plot_pdf['prediction'], best_plot_pdf['residual'], s=8, alpha=0.35)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Prediction'); axes[0].set_ylabel('Prediction − actual')
axes[0].set_title(f'{best_name}: residuals')
axes[1].hist(best_plot_pdf['residual'], bins=60, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Распределение остатков')
axes[1].set_xlabel('Residual')
axes[2].scatter(best_plot_pdf[TARGET], best_plot_pdf['prediction'], s=8, alpha=0.35)
lo = float(min(best_plot_pdf[TARGET].min(), best_plot_pdf['prediction'].min()))
hi = float(max(best_plot_pdf[TARGET].max(), best_plot_pdf['prediction'].max()))
axes[2].plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Prediction')
axes[2].set_title('Actual vs predicted')
plt.tight_layout()
plt.savefig(ART_DIR / ('actual_vs_predicted_outliers.png' if TASK == 'outliers' else 'actual_vs_predicted.png'), bbox_inches='tight')
plt.savefig(ART_DIR / 'residuals.png', bbox_inches='tight')
if TASK == 'walmart':
    plt.savefig(ART_DIR / 'residuals_importance.png', bbox_inches='tight')
plt.show()

compare_pdf = pd.DataFrame(results).T.sort_values(HERO_METRIC)
fig, ax = plt.subplots(figsize=(11, 5))
compare_pdf[['mae', 'medae', 'rmse']].plot(kind='bar', ax=ax)
ax.set_title('Сравнение моделей: MAE / MedAE / RMSE')
ax.set_ylabel('Ошибка в единицах target')
ax.set_xlabel('Модель')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(ART_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()

# Диагностики строятся распределённо по полному test для победителя.
err_df = (best_pred_sdf.select(F.col(TARGET).alias('_y'), F.col('prediction').alias('_p'))
          .withColumn('_err', F.col('_p') - F.col('_y'))
          .withColumn('_abs_err', F.abs(F.col('_p') - F.col('_y'))))
mae_best = float(results[best_name]['mae'])
residual_stats = err_df.agg(
    F.avg('_err').alias('bias'),
    F.expr('percentile_approx(_err, 0.5, 10000)').alias('median_residual'),
    F.stddev_pop('_err').alias('std_residual'),
    F.skewness('_err').alias('skew'),
    F.kurtosis('_err').alias('kurtosis'),
    (F.avg(F.when(F.col('_abs_err') > F.lit(2 * mae_best), 1.0).otherwise(0.0)) * 100).alias('pct_beyond_2mae'),
    (F.avg(F.when(F.col('_abs_err') > F.lit(3 * mae_best), 1.0).otherwise(0.0)) * 100).alias('pct_beyond_3mae'),
).first().asDict()
residual_stats = {k: (float(v) if v is not None else 0.0) for k, v in residual_stats.items()}

# Feature importance деревьев: значения остаются в исходной системе признаков.
feature_importance = {}
for model_name in ['DecisionTree', 'RandomForest', 'GradBoost']:
    if model_name not in fitted:
        continue
    tree_model = fitted[model_name].stages[-1]
    if hasattr(tree_model, 'featureImportances'):
        vals = tree_model.featureImportances.toArray().tolist()
        feature_importance[model_name] = [
            {'name': feature, 'value': float(value)}
            for feature, value in zip(FEATURES, vals)
        ]

# Дополнительный график важности RF и GBT, если они доступны.
if feature_importance:
    first_name = 'RandomForest' if 'RandomForest' in feature_importance else next(iter(feature_importance))
    fi_pdf = pd.DataFrame(feature_importance[first_name]).sort_values('value')
    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(fi_pdf))))
    ax.barh(fi_pdf['name'], fi_pdf['value'])
    ax.set_title(f'Feature importance — {first_name}')
    plt.tight_layout()
    plt.savefig(ART_DIR / 'feature_importance.png', bbox_inches='tight')
    plt.show()


## 6. Кросс-валидация

Только Random Forest и GBT, три фолда. Это даёт проверку устойчивости двух нелинейных ансамблей и ограничивает число дополнительных запусков обучения.

In [ ]:
# Трёхфолдовая CV только для двух ансамблей. Полный перебор всех шести моделей
# увеличил бы число обучений в 3 раза без пропорциональной пользы для этого демо.
cv_fraction = min(1.0, (CV_SAMPLE_ROWS * 1.25) / max(N_TRAIN, 1))
cv_source = (train_df.sample(withReplacement=False, fraction=cv_fraction, seed=SEED + 7)
             .limit(CV_SAMPLE_ROWS).cache())
cv_source_count = cv_source.count()
cv_folds = cv_source.randomSplit([1.0] * CV_FOLDS, seed=SEED + 17)
print(f'CV dataset: {cv_source_count:,} rows (cap {CV_SAMPLE_ROWS:,})')
cv_rows = []
for model_name in CV_MODELS:
    fold_metrics = []
    for fold_idx in range(CV_FOLDS):
        valid_fold = cv_folds[fold_idx]
        train_fold = None
        for j, fold in enumerate(cv_folds):
            if j != fold_idx:
                train_fold = fold if train_fold is None else train_fold.unionByName(fold)
        fold_model = make_pipeline(model_name).fit(train_fold)
        fold_result = metrics_from_spark(fold_model.transform(valid_fold))
        fold_metrics.append(fold_result)
        print(f'CV {model_name}: fold {fold_idx + 1}/{CV_FOLDS} — '
              f'{HERO_METRIC}={fold_result[HERO_METRIC]:.4f}')
    keys = sorted(set.intersection(*(set(x.keys()) for x in fold_metrics)))
    row = {'model': model_name}
    for key in keys:
        vals = np.asarray([x[key] for x in fold_metrics], dtype=float)
        row[f'{key}_mean'] = float(vals.mean())
        row[f'{key}_std'] = float(vals.std(ddof=0))
    cv_rows.append(row)
cv_pdf = pd.DataFrame(cv_rows).set_index('model').round(4)
cv_source.unpersist()
cv_pdf


## 7. Сохранение Spark-модели и dashboard report.json

Сохраняется `PipelineModel` в Spark-формате, проверяется reload, затем формируются CSV/JSON-метрики, `report.json` и общий `artifacts/index.json`. Списки предсказаний в отчёте ограничены 5 000 строками.

In [ ]:
# Сохраняем Spark-модель в нативном формате Spark ML и собираем dashboard-артефакты.
from pyspark.ml import PipelineModel

model_path = MODEL_DIR / f'{TASK}_spark_model'
model_meta_path = MODEL_DIR / f'{TASK}_spark_model_meta.json'
if best_name == 'BaselineMedian':
    # Защитная ветка: baseline может стать победителем на другом варианте данных.
    model_path = None
else:
    fitted[best_name].write().overwrite().save(str(model_path))
    print('Spark model saved:', model_path)

# Smoke-test загрузки модели. Spark сохраняет PipelineModel, не joblib/sklearn pickle.
if model_path is not None:
    reloaded_model = PipelineModel.load(str(model_path))
    reloaded_metrics = metrics_from_spark(reloaded_model.transform(test_df))
    for metric_name in ['mae', 'rmse', 'r2', 'mape']:
        assert abs(reloaded_metrics[metric_name] - results[best_name][metric_name]) < 1e-6, (
            f'Reload mismatch for {metric_name}: {reloaded_metrics[metric_name]} vs {results[best_name][metric_name]}'
        )
    print('Reload check: OK')

metrics_for_csv = pd.DataFrame(results).T
metrics_for_csv.to_csv(ART_DIR / 'metrics_all_models.csv', index_label='model')
(ART_DIR / 'metrics_all_models.json').write_text(
    json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
cv_pdf.to_csv(ART_DIR / 'cross_validation.csv', index_label='model')

# Отношения метрик полезны для определения хвостов ошибок.
diag_rows = []
for name, vals in results.items():
    row = {
        'model': name,
        'RMSE/MAE': vals['rmse'] / max(vals['mae'], 1e-9),
        'MAE/MedAE': vals['mae'] / max(vals['medae'], 1e-9),
        'MAPE,%': vals['mape'],
        'R²': vals['r2'],
        '|Bias|': abs(vals['bias']),
        'MaxErr': vals['max_error'],
    }
    if 'rmsle' in vals:
        row['RMSLE'] = vals['rmsle']
    diag_rows.append(row)
diag_pdf = pd.DataFrame(diag_rows).set_index('model').round(4)
diag_pdf.to_csv(ART_DIR / 'diagnostics.csv')

# Сырые коэффициенты Ridge для совместимости с прежней структурой feature_importance.
if TASK in {'housing', 'outliers'} and 'Ridge' in fitted:
    ridge_stage = fitted['Ridge'].stages[-1]
    ridge_coefficients = ridge_stage.coefficients.toArray().tolist()
    if len(ridge_coefficients) == len(FEATURES):
        feature_importance['Ridge_coef'] = [
            {'name': feature, 'value': float(value)}
            for feature, value in zip(FEATURES, ridge_coefficients)
        ]

# Единый детерминированный набор строк для графиков и report.json, максимум 5 000 строк.
export_test = test_df.orderBy('_row_id').limit(MAX_REPORT_ROWS).cache()
export_n = export_test.count()
export_columns = ['_row_id'] + FEATURES + [TARGET] + ([c for c in ['Date'] if c in export_test.columns])
export_pdf = export_test.select(*list(dict.fromkeys(export_columns))).orderBy('_row_id').toPandas()
export_predictions = {}
for name in MODEL_NAMES:
    pred_pdf = predict_with(name, export_test).select('_row_id', 'prediction').orderBy('_row_id').toPandas()
    export_predictions[name] = pred_pdf['prediction'].astype(float).tolist()

# Статистики признаков и целевой — одна агрегирующая Spark-задача.
def spark_stats_for(cols):
    exprs = []
    for c in cols:
        exprs.extend([
            F.min(F.col(c)).alias(f'{c}__min'),
            F.max(F.col(c)).alias(f'{c}__max'),
            F.avg(F.col(c)).alias(f'{c}__mean'),
            F.stddev_pop(F.col(c)).alias(f'{c}__std'),
            F.percentile_approx(F.col(c), 0.5, 10000).alias(f'{c}__median'),
        ])
    row = data_sdf.agg(*exprs).first().asDict()
    return {c: {k: float(row[f'{c}__{k}'] or 0.0) for k in ['min', 'max', 'mean', 'std', 'median']} for c in cols}

feature_stats = spark_stats_for(FEATURES)
target_stats = spark_stats_for([TARGET])[TARGET]

# Пары/имена метрик лучшей модели и отдельные победители по каждой метрике.
metric_keys = ['mae', 'rmse'] + (['rmsle'] if TASK == 'walmart' else []) + ['mape', 'r2', 'medae', 'max_error', 'evs']
by_metric = {}
for metric in metric_keys:
    candidates = [(name, values[metric]) for name, values in results.items() if metric in values and np.isfinite(values[metric])]
    if candidates:
        by_metric[metric] = (max(candidates, key=lambda item: item[1])[0] if metric == 'r2'
                             else min(candidates, key=lambda item: item[1])[0])
by_metric['bias'] = min(results, key=lambda name: abs(results[name]['bias']))

models_block = []
for name, values in results.items():
    entry = {
        'name': name,
        'is_best': name == best_name,
        'is_best_by_rmse': name == best_by_rmse,
        'metrics': {k: float(v) for k, v in values.items()},
        'predictions': export_predictions[name],
        'diagnostics': {
            'rmse_over_mae': float(values['rmse'] / max(values['mae'], 1e-9)),
            'mae_over_medae': float(values['mae'] / max(values['medae'], 1e-9)),
            'abs_bias': float(abs(values['bias'])),
        },
    }
    if name in cv_pdf.index:
        entry['cv'] = {k: float(v) for k, v in cv_pdf.loc[name].items()}
    models_block.append(entry)

# Десять наименее ошибочных строк из экспортной выборки — только компактный сэмпл.
best_export_predictions = np.asarray(export_predictions[best_name], dtype=float)
export_actual = export_pdf[TARGET].astype(float).to_numpy()
export_errors = best_export_predictions - export_actual
closest_indices = np.argsort(np.abs(export_errors))[:min(10, len(export_errors))]
sample_predictions = []
for i in closest_indices:
    feature_values = {f: float(export_pdf.iloc[i][f]) for f in FEATURES}
    err = float(export_errors[i])
    sample_predictions.append({
        'features': feature_values,
        'actual': float(export_actual[i]),
        'predicted': float(best_export_predictions[i]),
        'error': err,
        'abs_error': float(abs(err)),
        'within_mae': bool(abs(err) < results[best_name]['mae']),
        'is_outlier': bool(abs(err) > 3 * results[best_name]['mae']),
    })

charts = {'eda': 'eda.png', 'model_comparison': 'model_comparison.png', 'residuals': 'residuals.png'}
if TASK == 'walmart':
    for chart_key, chart_file in [('residuals_importance', 'residuals_importance.png'), ('mae_vs_mape', 'mae_vs_mape.png'), ('mape_per_store', 'mape_per_store.png')]:
        if (ART_DIR / chart_file).exists():
            charts[chart_key] = chart_file
actual_chart = 'actual_vs_predicted_outliers.png' if TASK == 'outliers' else 'actual_vs_predicted.png'
charts['actual_vs_predicted' if TASK != 'outliers' else 'actual_vs_predicted_outliers'] = actual_chart
if (ART_DIR / 'feature_importance.png').exists():
    charts['feature_importance'] = 'feature_importance.png'

# Калькулятор оставлен в report для прежнего интерфейса: его поля — реальные признаки модели.
if TASK == 'housing':
    # Ridge-коэффициенты сохраняют совместимость с простым калькулятором в test.html.
    ridge_stage = fitted['Ridge'].stages[-1]
    ridge_coefficients = ridge_stage.coefficients.toArray().tolist()
    calculator = {
        'model': 'Ridge',
        'intercept': float(ridge_stage.intercept),
        'coefficients': {f: float(c) for f, c in zip(FEATURES, ridge_coefficients)},
    }
else:
    calculator = {
        'target': TARGET,
        'target_unit': TARGET_UNIT,
        'features': [
            {'name': f, 'min': feature_stats[f]['min'], 'max': feature_stats[f]['max'],
             'median': feature_stats[f]['median'], 'default': feature_stats[f]['median']}
            for f in FEATURES
        ],
        'model_format': 'spark_pipeline' if model_path is not None else 'constant_baseline',
        'model_path': str(model_path.relative_to(PROJECT)) if model_path is not None else None,
    }

headline = {
    'best_model': best_name,
    'best_metric': HERO_METRIC,
    'best_metric_value': float(results[best_name][HERO_METRIC]),
    'verdict': f'Лучшая по {HERO_METRIC.upper()} — {best_name}.',
    'runner_up_by_rmse': {
        'model': best_by_rmse,
        'rmse': float(results[best_by_rmse]['rmse']),
    },
    'diagnostic_verdict': {
        'rmse_over_mae': float(results[best_name]['rmse'] / max(results[best_name]['mae'], 1e-9)),
        'mae_over_medae': float(results[best_name]['mae'] / max(results[best_name]['medae'], 1e-9)),
        'flag': 'outliers_present' if results[best_name]['rmse'] / max(results[best_name]['mae'], 1e-9) > 1.5 else 'clean',
    },
}
if TASK == 'outliers':
    headline['huber_vs_linear'] = {
        'huber_medae': results.get('Huber', {}).get('medae'),
        'huber_rmse': results.get('Huber', {}).get('rmse'),
        'best_linear_medae': min((results[n]['medae'] for n in ['Linear', 'Ridge'] if n in results), default=None),
        'best_linear_rmse': min((results[n]['rmse'] for n in ['Linear', 'Ridge'] if n in results), default=None),
    }

report = {
    'meta': {
        'task': TASK,
        'title': TITLE,
        'subtitle': {
            'housing': 'Spark ML · распределённая регрессия на California Housing',
            'walmart': 'Spark ML · разные масштабы продаж → MAPE как основная метрика',
            'outliers': 'Spark ML · выбросы в цене и устойчивость MedAE',
        }[TASK],
        'dataset': DATASET_NAME,
        'target': TARGET,
        'target_unit': TARGET_UNIT,
        'target_format': 'money' if TASK in ['walmart', 'outliers'] else 'number',
        'features': FEATURES,
        'seed': SEED,
        'generated_at': datetime.now().isoformat(),
        'n_samples': int(N_SAMPLES),
        'n_features': int(len(FEATURES)),
        'n_train': int(N_TRAIN),
        'n_test': int(N_TEST),
        'n_points_in_json': int(export_n),
        'target_stats': target_stats,
        'narrative': {
            'housing': {
                'context': 'Предсказываем медианную стоимость дома в округе Калифорнии.',
                'hero_metric': 'MAE',
                'main_conclusion': 'Сравниваем линейные модели и деревья; MAE показывает типичный абсолютный промах, R² — объяснённую дисперсию.',
            },
            'walmart': {
                'context': 'Продажи по магазинам имеют разный масштаб; абсолютную ошибку дополняем MAPE и RMSLE.',
                'hero_metric': 'MAPE',
                'why_not_mae': 'MAE в долларах сильнее отражает большие магазины; MAPE сравнивает относительную ошибку.',
                'main_conclusion': 'Для бизнес-сравнения магазинов ориентируемся на MAPE, сохраняя MAE/RMSE для оценки абсолютного риска.',
            },
            'outliers': {
                'context': 'Цена алмаза имеет тяжёлый правый хвост; редкие дорогие камни сильнее влияют на RMSE.',
                'hero_metric': 'MedAE',
                'main_conclusion': 'Сопоставляем MedAE, MAE и RMSE: метрика зависит от того, важна типичная или квадратичная ошибка.',
                'rules': {
                    'rmse_over_mae_norm': '< 1.3 → ошибки ближе к однородным',
                    'rmse_over_mae_outliers': '> 1.5 → хвост крупных ошибок заметен',
                    'mae_over_medae_heavy_tail': '> 1.8 → медиана ошибки существенно ниже среднего',
                },
            },
        }[TASK],
    },
    'headline': headline,
    'data': {
        'y_test': [float(v) for v in export_actual],
        'feature_names': FEATURES,
        'feature_stats': feature_stats,
        'n_test_full': int(N_TEST),
    },
    'calculator': calculator,
    'models': models_block,
    'by_metric': by_metric,
    'residual_diagnostics': residual_stats,
    'feature_importance': feature_importance,
    'sample_predictions': sample_predictions,
    'charts': charts,
    'artifacts': {
        'metrics_csv': 'metrics_all_models.csv',
        'metrics_json': 'metrics_all_models.json',
        'diagnostics_csv': 'diagnostics.csv',
        'cv_csv': 'cross_validation.csv',
        'model': str(model_path.relative_to(PROJECT)) if model_path is not None else None,
        'model_meta': str(model_meta_path.relative_to(PROJECT)),
    },
}

# Walmart: сохраняем календарные оси и компактные недельные ряды для всех магазинов.
if TASK == 'walmart':
    def _iso_date(value):
        if pd.isna(value):
            return None
        return pd.Timestamp(value).strftime('%Y-%m-%d')

    report['data']['dates_test'] = [_iso_date(v) for v in export_pdf['Date'].tolist()]
    report['data']['stores_test'] = [int(v) for v in export_pdf['Store'].tolist()]
    report['data']['stores_test'] = report['data']['stores_test'][:export_n]
    train_export_pdf = (train_df.orderBy('_row_id').limit(MAX_REPORT_ROWS)
        .select('Date', 'Store').toPandas())
    report['data']['dates_train'] = [_iso_date(v) for v in train_export_pdf['Date'].tolist()]
    report['data']['stores_train'] = [int(v) for v in train_export_pdf['Store'].tolist()]
    report['data']['holidays'] = [
        {'date': '2010-02-12', 'icon': '🏈', 'label': 'Super Bowl'},
        {'date': '2010-09-10', 'icon': '🇺🇸', 'label': 'Labor Day'},
        {'date': '2010-11-26', 'icon': '🦃', 'label': 'Thanksgiving'},
        {'date': '2010-12-31', 'icon': '🎄', 'label': 'Christmas'},
        {'date': '2011-02-11', 'icon': '🏈', 'label': 'Super Bowl'},
        {'date': '2011-09-09', 'icon': '🇺🇸', 'label': 'Labor Day'},
        {'date': '2011-11-25', 'icon': '🦃', 'label': 'Thanksgiving'},
        {'date': '2011-12-30', 'icon': '🎄', 'label': 'Christmas'},
        {'date': '2012-02-10', 'icon': '🏈', 'label': 'Super Bowl'},
        {'date': '2012-09-07', 'icon': '🇺🇸', 'label': 'Labor Day'},
        {'date': '2012-11-23', 'icon': '🦃', 'label': 'Thanksgiving'},
        {'date': '2012-12-28', 'icon': '🎄', 'label': 'Christmas'},
        {'date': '2013-02-08', 'icon': '🏈', 'label': 'Super Bowl'},
    ]
    store_medians = (data_sdf.groupBy('Store')
        .agg(F.percentile_approx(TARGET, 0.5, 10000).alias('median_sales')))
    spread = store_medians.agg(F.min('median_sales').alias('median_min'),
                                F.max('median_sales').alias('median_max')).first().asDict()
    min_med = float(spread['median_min'] or 0.0)
    max_med = float(spread['median_max'] or 0.0)
    report['meta']['store_spread'] = {
        'ratio_max_min': float(max_med / min_med) if abs(min_med) > 1e-9 else None,
        'median_min': min_med,
        'median_max': max_med,
    }
    # Не больше 500 точек на магазин — обычно этого хватает для всех недель 2010–2012.
    series_window = Window.partitionBy('Store').orderBy('Date')
    series_pdf = (data_sdf.withColumn('_rn', F.row_number().over(series_window))
        .filter(F.col('_rn') <= 500)
        .select('Store', F.date_format('Date', 'yyyy-MM-dd').alias('date'), TARGET)
        .orderBy('Store', 'date').toPandas())
    series_by_store = {}
    for store_id, sub in series_pdf.groupby('Store', sort=True):
        series_by_store[str(int(store_id))] = {
            'dates': sub['date'].astype(str).tolist(),
            'sales': [float(v) for v in sub[TARGET].tolist()],
        }
    date_stats = data_sdf.agg(F.min('Date').alias('start'), F.max('Date').alias('end'),
                              F.countDistinct('Date').alias('n_weeks'),
                              F.countDistinct('Store').alias('n_stores')).first().asDict()
    report['series_by_store'] = series_by_store
    report['meta']['series_span'] = {
        'start': _iso_date(date_stats['start']),
        'end': _iso_date(date_stats['end']),
        'n_weeks': int(date_stats['n_weeks'] or 0),
        'n_stores': int(date_stats['n_stores'] or 0),
    }
    # Считаем per-store test метрики распределённо; в JSON попадут только группы магазинов.
    per_store_sdf = (best_pred_sdf
        .withColumn('_ae', F.abs(F.col('prediction') - F.col(TARGET)))
        .withColumn('_ape', F.when(F.abs(F.col(TARGET)) > 1e-9, F.col('_ae') / F.abs(F.col(TARGET)) * 100))
        .groupBy('Store')
        .agg(F.count('*').alias('n_weeks'), F.avg(TARGET).alias('avg_sales'),
             F.avg('_ae').alias('mae'), F.sqrt(F.avg(F.pow(F.col('prediction') - F.col(TARGET), 2))).alias('rmse'),
             F.avg('_ape').alias('mape'))
        .orderBy('Store'))
    report['per_store'] = [{
        'store': int(row['Store']), 'n_weeks': int(row['n_weeks']),
        'avg_sales': float(row['avg_sales']), 'mae': float(row['mae']),
        'rmse': float(row['rmse']), 'mape': float(row['mape'] or 0.0),
    } for row in per_store_sdf.collect()]

report_path = ART_DIR / 'report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

model_meta = {
    'task': TASK,
    'dataset': DATASET_NAME,
    'format': 'pyspark.ml.PipelineModel' if model_path is not None else 'constant_baseline',
    'model': best_name,
    'model_path': str(model_path.relative_to(PROJECT)) if model_path is not None else None,
    'features': FEATURES,
    'target': TARGET,
    'target_unit': TARGET_UNIT,
    'n_samples': int(N_SAMPLES),
    'seed': SEED,
    'spark_version': spark.version,
    'metrics': results[best_name],
    'all_models_metrics': results,
    'saved_at': datetime.now().isoformat(),
}
if best_name == 'BaselineMedian':
    model_meta['constant_value'] = float(TRAIN_MEDIAN)
model_meta_path.write_text(json.dumps(model_meta, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

# Обновляем только запись текущей задачи в общий индекс проекта.
index_path = PROJECT / 'artifacts' / 'index.json'
index_path.parent.mkdir(parents=True, exist_ok=True)
if index_path.exists():
    try:
        index = json.loads(index_path.read_text(encoding='utf-8'))
    except Exception:
        index = {'generated_at': None, 'tasks': []}
else:
    index = {'generated_at': None, 'tasks': []}
index.setdefault('tasks', [])
index['tasks'] = [item for item in index['tasks'] if item.get('id') != TASK]
index['tasks'].append({
    'id': TASK,
    'title': TITLE,
    'kind': 'regression',
    'best_model': best_name,
    'hero_metric': HERO_METRIC.upper(),
    'hero_value': float(results[best_name][HERO_METRIC]),
    'unit': '%' if HERO_METRIC == 'mape' else TARGET_UNIT,
    'report': str(report_path.relative_to(PROJECT)).replace('\\', '/'),
    'charts_dir': str(ART_DIR.relative_to(PROJECT)).replace('\\', '/'),
})
index['tasks'] = sorted(index['tasks'], key=lambda item: item.get('id', ''))
index['generated_at'] = datetime.now().isoformat()
index_path.write_text(json.dumps(index, indent=2, ensure_ascii=False), encoding='utf-8')

print('\nREPORT:', report_path)
print('INDEX :', index_path)
print('BEST  :', best_name, HERO_METRIC, round(results[best_name][HERO_METRIC], 5))
print('MODELS:', list(results))
print('JSON points:', export_n, '(full test rows:', N_TEST, ')')
print('Artifacts:')
for path in sorted(ART_DIR.iterdir()):
    print(f'  {path.name:34s} {path.stat().st_size:>10,} bytes')


## Итог и ограничения

- Все шесть вариантов сравниваются на одном Spark train/test split 80/20.
- MAE, RMSE, MAPE, MedAE, R², Bias, MaxErr и EVS считаются Spark SQL-агрегациями; MedAE использует `approxQuantile` с относительной ошибкой 0.1%.
- Трёхфолдовая CV запускается только для RandomForest и GBT. Размер деревьев ограничен (`40` деревьев/итераций, глубина до `8`/`4`).
- В report JSON записывается максимум 5 000 согласованных строк предсказаний на модель. Метрики всегда считаются по полному тестовому набору.
- Spark-модель сохраняется в нативном формате `pyspark.ml.PipelineModel` в `models/regression/housing/`; это не `.joblib`-файл scikit-learn.
- Если Spark Standalone не запущен, ноутбук использует `local[2]`. Для distributed режима включите профиль Spark в Docker Compose.

**Основная метрика:** `MAE`. Победитель выбирается автоматически по полной тестовой выборке.


In [ ]:
print('Task:', TASK)
print('Spark:', spark.version)
print('Rows:', N_SAMPLES, '| train:', N_TRAIN, '| test:', N_TEST)
print('Hero:', best_name, HERO_METRIC, results[best_name][HERO_METRIC])
print('Model paths are Spark-native; API code expecting sklearn .joblib must be adapted before loading these models.')
